# Module 03 — State, Memory & Recovery (Colab)

Two agent runs serve two different tenants while sharing one memory object. Tenant Beta's data leaks into tenant Alpha's briefing — written to a real Postgres database. You'll fix the isolation and add crash recovery.

**Run the cells top to bottom.** The setup cell takes a few minutes the first time.

## 1. Get the code and set up the environment

In [ ]:
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

In [ ]:
!bash setup.sh

In [ ]:
import os
os.environ['NOVA_LLM'] = 'ollama'
os.environ['OLLAMA_MODEL'] = 'llama3.2:1b'
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'
print('environment configured')

In [ ]:
!python preflight.py

## 2. Watch the cross-tenant leak

Watch the `memory.tenant` column: at the red rows, run-a (serving Alpha) is acting on memory that already says `beta`.

In [ ]:
!python modules/03_state/naive_state.py

See the leaked briefing row in the **real database**:

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, left(content,70) AS briefing FROM briefings;"

## 3. Fix it

The cell below is `your_fix.py`. Two tasks:
1. Make `IsolatedState` namespace memory by `run_id` (edit `get`/`set`).
2. Make `run_recoverable` skip work already completed before the kill.

Edit the cell, then re-run it to save.

In [ ]:
%%writefile modules/03_state/your_fix.py
from nova.agent import Agent, format_briefing_content
from nova.models import Briefing


class IsolatedState:
    def __init__(self) -> None:
        self._data: dict = {}

    def get(self, run_id: str) -> dict:
        # TODO: isolate by run_id instead of returning one shared dict
        return self._data

    def set(self, run_id: str, data: dict) -> None:
        # TODO: isolate by run_id instead of mutating one shared dict
        self._data.update(data)


def run_recoverable(agent: Agent, client_id: str, run_id: str, kill_after=None):
    # TODO: on resume, skip steps that already completed (don't redo the write)
    for checkpoint in agent.run_steps(client_id, run_id):
        if checkpoint == kill_after:
            return None
    working = agent.state.get(run_id)
    return Briefing(
        client_id=client_id,
        content=format_briefing_content(working.get('client_id', client_id), working.get('last_response', '')),
        obligations=agent.store.get_obligations(client_id),
    )

In [ ]:
!python -m pytest modules/03_state/test_state.py -v

## 4. See it land

In [ ]:
!python modules/03_state/compare.py